In [ ]:
# Lab type: review
# Course: DS101 — Introduction to Data Science
# Lesson: Reviewing AI-Generated Code — A Practical Checklist
# Task: Review AI-generated churn-preparation code against the lesson's five-point checklist.
# Run each check, answer the judgment question below it, then compare with the revealed answer.

# Lab: Reviewing AI-Generated Code

An AI assistant was asked to *"prepare the customers dataset for a churn classifier"*. Its code is below, the same code as in the lesson. It runs without errors and prints a reassuring message.

Your job isn't to find a fixed number of bugs. It's to review the code the way the lesson describes, running each of the five checks and deciding for yourself whether the result is acceptable:

1. Does the shape make sense?
2. Are missing values handled correctly?
3. Are column types what you expect?
4. Is the split respected?
5. Does the output match what you intended?

Write your answer to each question before you open the reveal.

## Setup: build the dataset

In [ ]:
!pip install pandas numpy scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd

# In the lesson this was read from customers.csv. Here we build it so the lab runs anywhere.
rng = np.random.default_rng(11)
n = 1_000

plan = rng.choice(["Basic", "Standard", "Premium"], size=n, p=[0.5, 0.35, 0.15]).astype(object)
plan[rng.random(n) < 0.04] = np.nan                  # some customers have no plan recorded
tenure_months = rng.integers(1, 72, n).astype(float)
tenure_months[rng.random(n) < 0.03] = np.nan         # and some have no tenure
monthly_spend = rng.normal(45, 15, n).clip(5).round(2)
churn_prob = 0.45 - tenure_months / 200
churned = np.where(rng.random(n) < np.nan_to_num(churn_prob, nan=0.3), "yes", "no")

customers = pd.DataFrame({
    "customer_id": np.arange(100_001, 100_001 + n),
    "region": rng.choice(["North", "South", "East", "West"], size=n),
    "plan": plan,
    "tenure_months": tenure_months,
    "monthly_spend": monthly_spend,
    "churned": churned,
})
customers.head()

## The AI-generated code

Run it as written. Then work through the five checks below.

In [ ]:
# Review this code — is it correct?
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = customers.copy()   # the AI's code started with: df = pd.read_csv("customers.csv")

# Encode categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

# Scale all numeric columns
scaler = StandardScaler()
df[df.select_dtypes(include="number").columns] = scaler.fit_transform(
    df.select_dtypes(include="number")
)

# Split into train and test
X = df.drop("churned", axis=1)
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Ready to train. X_train shape:", X_train.shape)

## Check 1: Does the shape make sense?

In [ ]:
print("Raw data:", customers.shape)
print("X_train: ", X_train.shape, "  X_test:", X_test.shape)
print("Feature columns:", list(X_train.columns))

> **Question:** The row counts add up. Is every column in `X_train` something a churn model should learn from? Which one would you remove, and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Check 1</summary>

**Rows: fine.** 800 + 200 = 1,000; nothing was dropped or duplicated.

**Columns: not fine.** `customer_id` is in the features. It's an identifier, not a property of the customer, so any pattern the model finds in it is coincidence (for example, newer IDs belonging to newer customers). Worse, the code scaled it as if it were a measurement. The shape check passes only if you stop at the row count. Always read the column list too.

**Fix:** drop identifier columns before modelling: `X = df.drop(columns=["churned", "customer_id"])`.

</details>

## Check 2: Are missing values handled correctly?

In [ ]:
print("Missing values in the raw data:")
print(customers.isna().sum())

print("\nMissing values in X_train after the AI's code:")
print(X_train.isna().sum())

print("\nWhat happened to customers with no plan? Encoded plan values:")
print(df.loc[customers["plan"].isna(), "plan"].unique())

> **Question:** The code never mentions missing values. What happened to the missing `plan` values, and to the missing `tenure_months` values? Which of the two will cause a visible problem later, and which one won't?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Check 2</summary>

**`plan`:** `LabelEncoder` quietly turned "missing" into one more integer code, so the model will treat "no plan recorded" as a fourth plan type. That may even be reasonable, but nobody decided it: it happened by accident and nothing in the output says so. **This one never causes a visible error**, which makes it the more dangerous of the two.

**`tenure_months`:** `StandardScaler` ignores NaN when it learns the mean and passes NaN through, so the missing values are still there after scaling. Most scikit-learn models refuse NaN, so this one **will** fail loudly when you call `fit`, just later than the line that caused it.

**Fix:** decide per column, before encoding: for example, an explicit `"Unknown"` plan, and a median imputer for tenure fitted on the training rows only (check 4).

</details>

## Check 3: Are column types what you expect?

In [ ]:
print("Target values before the AI's code:", list(customers["churned"].unique()))
print("Target values after the AI's code: ", np.unique(y_train.round(3)))
print()
print("plan codes:  ", np.sort(df["plan"].unique()).round(3))
print("LabelEncoder now remembers only:", list(le.classes_))

> **Question:** Three things in this output are wrong. (1) What happened to the target column? (2) What does encoding `Basic`/`Standard`/`Premium` as single numbers imply to a model? (3) Why can't this `le` object be used later to encode new customers?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Check 3</summary>

**(1) The target was encoded and then scaled.** `churned` was an `object` column ("yes"/"no"), so the loop label-encoded it to 0/1. That made it numeric, so the scaler then standardised it into two odd floats. A classifier needs class labels; most will reject a continuous target, or you'll end up predicting "-0.6". The target should be separated **before** any feature processing.

**(2) Label encoding implies order.** Basic=0, Premium=1, Standard=2 (alphabetical!) tells a linear model that Standard is "twice" Premium. For nominal categories use one-hot encoding (`OneHotEncoder`), which creates one 0/1 column per category.

**(3) One encoder, many columns.** The same `le` was refitted on each column in turn, so at the end it only knows the last column it saw (the target). You can't use it to encode `region` or `plan` for new customers. Use one encoder per column, or better, a `ColumnTransformer`.

</details>

## Check 4: Is the split respected?

In [ ]:
# The scaler was fitted before the split. What did it learn from?
print("Rows the scaler learned from:", int(np.max(scaler.n_samples_seen_)))
print("Rows in the training set:    ", len(X_train))

# What happens when a category appears that the encoder never saw?
train_plan_encoder = LabelEncoder().fit(customers["plan"].dropna())
try:
    train_plan_encoder.transform(["Enterprise"])
except ValueError as err:
    print("\nNew plan type at prediction time ->", err)

> **Question:** The scaler learned its mean and standard deviation from how many rows? Why does that make the test score less trustworthy? And what will happen in production the first time a customer signs up for a plan that didn't exist when the model was trained?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Check 4</summary>

**Leakage:** the scaler (and the encoder) learned from all 1,000 rows, including the 200 test rows. The test set is supposed to stand in for data the model has never seen; here it helped shape the preprocessing. With a scaler the effect is usually small, but it's the same mistake that makes target encoding or imputation leak badly, and it's the habit that matters.

**Unseen categories:** `LabelEncoder` raises `ValueError` on any value it didn't see during `fit`. In production a new plan type ("Enterprise") would crash the prediction step. `OneHotEncoder(handle_unknown="ignore")` handles this deliberately.

**Fix:** split first, then fit every encoder, imputer and scaler on the training rows only and apply it to the test rows with `transform`. A scikit-learn `Pipeline` makes this the default.

</details>

## Check 5: Does the output match what you intended?

In [ ]:
print(y_train.value_counts().round(3))
print()
print(X_train.describe().round(2))

> **Question:** If you'd only read the final `print` ("Ready to train"), would you have caught any of the problems above? Looking at this output, what single line would have told you immediately that something was wrong with the target?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Check 5</summary>

**"Ready to train" proved nothing.** It only shows the code ran. Every problem above survives to this point without an error.

**The giveaway:** `y_train.value_counts()` should show two labels (`no`/`yes`, or `0`/`1`) with counts. Instead it shows two unexplained decimals. One line of inspection on the target would have caught the biggest bug in two seconds. That's the lesson's point: run the code on real data and *look* at what comes out.

</details>

## A corrected version

Here is one correct way to write the same preparation step. It separates the target first, drops the identifier, splits before fitting anything, and lets a `ColumnTransformer` handle missing values, scaling and one-hot encoding using the training rows only. Run it and repeat checks 1, 2 and 5.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = customers.copy()
y = (df["churned"] == "yes").astype(int)              # target: separated first, as 0/1
X = df.drop(columns=["churned", "customer_id"])       # drop the target and the identifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric = ["tenure_months", "monthly_spend"]
categorical = ["region", "plan"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

X_train_ready = preprocess.fit_transform(X_train)   # fit on training rows only
X_test_ready = preprocess.transform(X_test)         # transform only

print("X_train_ready:", X_train_ready.shape, " X_test_ready:", X_test_ready.shape)
print("Missing values left:", int(np.isnan(X_train_ready).sum()))
print("Target values:", y_train.value_counts().to_dict())
print("Features:", list(preprocess.get_feature_names_out()))

> **Final question:** Which of the five checks found the most serious problem in the original code? If you could run only two checks on the next piece of AI-generated code you receive, which two would you pick?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Final question</summary>

There's no single right answer, but a strong one notices that **check 3 (types) and check 5 (inspect the output)** caught the most serious problem, the scaled target, and that check 5 is the cheapest. A good pair for a quick review is **check 5** (look at the target and a few rows) plus **check 4** (was anything fitted before the split?), because those two problems never raise an error on their own.

</details>